In [ ]:
# Подключение к MLflow
import mlflow
import os

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"

mlflow.set_tracking_uri("http://localhost:5000")

c:\Users\vasav\anaconda3\envs\final_project_start_ml_py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Загрузка модели по тегу PRD
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    experiment_ids=["1"],
    filter_string="tags.stage = 'PRD'"
)
prd_run = runs[0]
run_id = prd_run.info.run_id
print(f"Run ID: {run_id}")
print(f"Run name: {prd_run.info.run_name}")

local_dir = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="models"
)
print(f"Артефакты скачаны в: {local_dir}")

Run ID: 5b4041d9f1b8443f9ecfde10a0a30368
Run name: catboost_tuned_final


Артефакты скачаны в: C:\Users\vasav\AppData\Local\Temp\tmpgkd8yf_c\models


In [4]:
# Загрузка моделей и списка признаков
from catboost import CatBoostClassifier

with open(f"{local_dir}/selected_features.txt") as f:
    SELECTED_FEATURES = f.read().strip().split("\n")
print(f"Признаков: {len(SELECTED_FEATURES)}")

SUPERCLASSES = ["CD", "HYP", "MI", "NORM", "STTC"]
models = {}
for sc in SUPERCLASSES:
    models[sc] = CatBoostClassifier()
    models[sc].load_model(f"{local_dir}/catboost_{sc}.cbm")

print("Все модели загружены")

Признаков: 176
Все модели загружены


In [ ]:
# Подготовка тестовых данных
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import welch

DATA_PATH = "../datasets/clean_ptbxl_with_ecg_n_diagnostic_superclass.pkl"
df = pd.read_pickle(DATA_PATH)

df = df.drop(['pacemaker', 'extra_beats', 'infarction_stadium1', 'infarction_stadium2'], axis=1)

def fill_rand_based_q(df, cols=['height', 'weight'], group='sex'):
    df = df.copy()
    np.random.seed(42)
    for col in cols:
        q = df.groupby(group)[col].quantile([0.1, 0.9]).unstack()
        for s in q.index:
            low, high = q.loc[s, 0.1], q.loc[s, 0.9]
            mask = (df[group] == s) & (df[col].isna())
            if pd.isna(low) or pd.isna(high) or low == high:
                continue
            df.loc[mask, col] = np.random.uniform(low, high, mask.sum())
    return df

df = fill_rand_based_q(df)
df['heart_axis'] = df.heart_axis.fillna('NO_DATA')
df['age'] = df['age'].replace(300, 98)
df = df[df['height'] >= 90]
df = df[df['weight'] > 25]
df = df[df['diagnostic_superclass'].apply(lambda x: len(x) > 0)]

unique_diag = sorted({lbl for row in df['diagnostic_superclass'] for lbl in row})
for lbl in unique_diag:
    df[lbl] = df['diagnostic_superclass'].apply(lambda x: int(lbl in x))

mask_bad = (df['NORM'] == 1) & ((df['CD'] == 1) | (df['HYP'] == 1) | (df['MI'] == 1) | (df['STTC'] == 1))
df = df[~mask_bad].reset_index(drop=True)
df = pd.get_dummies(df, columns=['heart_axis'], prefix='heart_axis')


def extract_ecg_features(ecg):
    features = {}
    all_stds, all_ranges, all_energies = [], [], []

    for lead_idx in range(ecg.shape[1]):
        lead = ecg[:, lead_idx].astype(np.float64)
        p = f'lead{lead_idx + 1}'

        features[f'{p}_mean'] = np.mean(lead)
        features[f'{p}_median'] = np.median(lead)
        features[f'{p}_std'] = np.std(lead)
        features[f'{p}_min'] = np.min(lead)
        features[f'{p}_max'] = np.max(lead)
        features[f'{p}_range'] = np.max(lead) - np.min(lead)
        features[f'{p}_rms'] = np.sqrt(np.mean(lead ** 2))
        features[f'{p}_skew'] = float(skew(lead))
        features[f'{p}_kurt'] = float(kurtosis(lead))

        energy = np.sum(lead ** 2)
        features[f'{p}_energy'] = energy
        features[f'{p}_norm_energy'] = energy / len(lead)

        freqs, psd = welch(lead, fs=500)
        psd_norm = psd / (np.sum(psd) + 1e-12)
        features[f'{p}_dom_freq'] = freqs[np.argmax(psd)]
        features[f'{p}_spec_entropy'] = float(-np.sum(psd_norm * np.log(psd_norm + 1e-12)))

        lf_mask = (freqs >= 0.04) & (freqs <= 0.15)
        hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
        lf_e = np.sum(psd[lf_mask])
        hf_e = np.sum(psd[hf_mask])
        features[f'{p}_lf_energy'] = lf_e
        features[f'{p}_hf_energy'] = hf_e
        features[f'{p}_lf_hf_ratio'] = lf_e / (hf_e + 1e-6)

        p25, p75 = np.percentile(lead, [25, 75])
        features[f'{p}_p25'] = p25
        features[f'{p}_p75'] = p75
        features[f'{p}_iqr'] = p75 - p25
        features[f'{p}_mad'] = np.mean(np.abs(lead - np.mean(lead)))

        signs = np.sign(lead)
        signs[signs == 0] = 1
        features[f'{p}_zcr'] = np.sum(np.diff(signs) != 0) / len(lead)

        diff = np.diff(lead)
        features[f'{p}_mean_abs_diff'] = np.mean(np.abs(diff))
        features[f'{p}_std_diff'] = np.std(diff)
        features[f'{p}_max_abs_diff'] = np.max(np.abs(diff))

        if np.std(lead) > 1e-6:
            features[f'{p}_autocorr1'] = np.corrcoef(lead[:-1], lead[1:])[0, 1]
        else:
            features[f'{p}_autocorr1'] = 0.0

        features[f'{p}_cv'] = np.std(lead) / (np.abs(np.mean(lead)) + 1e-6)

        hf2_mask = (freqs >= 0.4) & (freqs <= 2.0)
        features[f'{p}_hf2_energy'] = np.sum(psd[hf2_mask])

        total_e = np.sum(psd) + 1e-12
        features[f'{p}_lf_rel'] = lf_e / total_e
        features[f'{p}_hf_rel'] = hf_e / total_e

        all_stds.append(np.std(lead))
        all_ranges.append(np.max(lead) - np.min(lead))
        all_energies.append(energy)

    cross_lead_pairs = [(0, 1), (0, 2), (1, 2), (3, 4), (6, 10), (7, 11)]
    cross_lead_names = ['I_II', 'I_III', 'II_III', 'aVR_aVL', 'V1_V5', 'V2_V6']
    for (i, j), name in zip(cross_lead_pairs, cross_lead_names):
        lead_i = ecg[:, i].astype(np.float64)
        lead_j = ecg[:, j].astype(np.float64)
        if np.std(lead_i) > 1e-6 and np.std(lead_j) > 1e-6:
            features[f'corr_{name}'] = np.corrcoef(lead_i, lead_j)[0, 1]
        else:
            features[f'corr_{name}'] = 0.0

    features['global_mean_std'] = np.mean(all_stds)
    features['global_max_range'] = np.max(all_ranges)
    features['global_energy_var'] = np.var(all_energies)
    features['global_energy_total'] = np.sum(all_energies)

    return features


feature_rows = []
for idx, row in df.iterrows():
    feature_rows.append(extract_ecg_features(row['ecg_signals']))

ecg_df = pd.DataFrame(feature_rows)
df_features = pd.concat([
    df.drop(columns=['ecg_signals']).reset_index(drop=True),
    ecg_df.reset_index(drop=True)
], axis=1)
df_features['bmi'] = df_features['weight'] / ((df_features['height'] / 100) ** 2)
df_features['strat_fold'] = df_features['strat_fold'].astype(int)

df_test = df_features[df_features['strat_fold'].between(9, 10)].copy()
X_test = df_test[SELECTED_FEATURES]

print(f"X_test shape: {X_test.shape}")

X_test shape: (3184, 176)


In [6]:
# Тестовый предикт
preds = {sc: models[sc].predict(X_test.values[:5]) for sc in models}
for sc, p in preds.items():
    print(f"{sc}: {p}")

CD: [0 1 1 0 1]
HYP: [0 0 1 0 0]
MI: [0 1 1 0 0]
NORM: [1 0 0 1 0]
STTC: [0 1 1 0 0]
